# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json_str = json.dumps(dataset.metadata.to_json(), indent=2)
print(metadata_json_str)
print(f"\nDataset Title: {dataset.metadata.name}")
print(f"\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their field ids
print("Available RecordSets (with @id):\n")
record_sets = dataset.metadata.get("recordSet", [])  # fallback: empty list if not present
if not record_sets:
    print("No record sets found in metadata. Checking distribution for files.")
    # Try inferring record set info from distribution
    distributions = dataset.metadata.get("distribution", [])
    for dist in distributions:
        print(dist)
else:
    for rset in record_sets:
        rset_id = rset.get("@id", None)
        rset_name = rset.get("name", None)
        print(f" - {rset_name} [@id={rset_id}]")
        fields = rset.get("field", [])
        for fld in fields:
            print(f"    - Field [@id={fld.get('@id', fld)}]")

In [ ]:
# Display example records for the first available record set (by @id)
record_sets = dataset.metadata.get("recordSet", [])
if record_sets:
    record_set_id = record_sets[0]["@id"]
    print(f"\nSample records from record set @id={record_set_id}:")
    for i, x in enumerate(dataset.records(record_set=record_set_id)):
        print(x)
        if i >= 2:
            break
else:
    print("No record sets available in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set using their @id
record_sets_list = dataset.metadata.get("recordSet", [])
record_set_ids = [rset["@id"] for rset in record_sets_list]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id={record_set_id} with shape {df.shape}")

# Display first 5 columns and sample for the first record set
if record_set_ids:
    first_rsid = record_set_ids[0]
    print(f"\nColumns in record set @id={first_rsid}:\n{dataframes[first_rsid].columns.tolist()}")
    display(dataframes[first_rsid].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
# Choose numeric and group fields using @id field names (replace as appropriate for your dataset after running previous cells)

# Example: Find a numeric field in the first DataFrame
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Try to automatically detect numeric columns (float or int)
numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
if not numeric_cols:
    # Try to convert possible columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()

if numeric_cols:
    numeric_field = numeric_cols[0]  # choose first numeric field
    print(f"Selected numeric field: {numeric_field}")
    
    # Filtering for values above a threshold
    threshold = df[numeric_field].mean() if df[numeric_field].notna().sum()>0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field (choose a category-like field)
    group_fields = df.select_dtypes(include=['object']).columns.tolist()
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for plotting.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* This notebook demonstrates how to load and explore a dataset with the `mlcroissant` library, identifying available record sets, extracting fields by `@id`, performing EDA and visualizations.
* For in-depth analysis, examine the field names, types, and relationships carefully, referring to their `@id` to ensure reproducible results.
* The FAIR^2 schema allows for rich metadata and standardized data extraction. Extend this notebook for your research needs!